# Whale Fills Analysis
Load all 139 whale fill files into a single pandas DataFrame and explore the data.

In [ ]:
import json
from pathlib import Path
from datetime import datetime, timezone
import pandas as pd

DATA_DIR = Path("data")

records = []
skipped = []

for f in sorted(DATA_DIR.glob("*.fills.json")):
    address = f.stem.replace(".fills", "")
    with open(f) as fp:
        fills = json.load(fp)
    if not fills:
        skipped.append(address)
        continue
    for fill in fills:
        fill["address"] = address
        records.append(fill)

print(f"Loaded {len(records):,} fill records from {len(records) > 0 and 'multiple' or 0} wallets")
print(f"Skipped {len(skipped)} wallets with no fills")

In [ ]:
df = pd.DataFrame(records)

# --- Type casting ---
df["px"]         = pd.to_numeric(df["px"],         errors="coerce")  # price
df["sz"]         = pd.to_numeric(df["sz"],         errors="coerce")  # size (asset units)
df["closedPnl"]  = pd.to_numeric(df["closedPnl"],  errors="coerce")  # realized PnL
df["fee"]        = pd.to_numeric(df["fee"],         errors="coerce")  # fee in USDC
df["startPosition"] = pd.to_numeric(df["startPosition"], errors="coerce")  # position before fill

# --- Timestamps ---
df["time"] = pd.to_datetime(df["time"], unit="ms", utc=True)

# --- Derived columns ---
df["notional"] = df["px"] * df["sz"]   # USD value of the fill
df["date"]     = df["time"].dt.date     # plain date for grouping

# --- Rename for clarity ---
df = df.rename(columns={
    "coin":    "asset",
    "px":      "price",
    "sz":      "size",
    "dir":     "direction",
    "side":    "side",       # B=buy A=ask(sell)
    "crossed": "is_taker",
})

# --- Column order ---
cols = ["address", "time", "date", "asset", "direction", "side",
        "price", "size", "notional", "closedPnl", "fee",
        "startPosition", "is_taker", "hash", "oid", "tid", "cloid", "feeToken", "twapId"]
# keep only cols that exist (cloid is absent on some wallets)
cols = [c for c in cols if c in df.columns]
df = df[cols].sort_values("time").reset_index(drop=True)

print(f"DataFrame shape: {df.shape}")
print(f"\nDate range: {df['time'].min().date()} -> {df['time'].max().date()}")
print(f"Unique wallets: {df['address'].nunique()}")
print(f"Unique assets:  {df['asset'].nunique()}")
df.dtypes

In [ ]:
# Preview the first few rows
df.head(10)

In [ ]:
# --- Quick summary stats ---
print("=== Direction breakdown ===")
print(df["direction"].value_counts())

print("\n=== Top 15 most traded assets (by fill count) ===")
print(df["asset"].value_counts().head(15))

print("\n=== Top 15 most traded assets (by total notional USD) ===")
print(df.groupby("asset")["notional"].sum().sort_values(ascending=False).head(15).apply(lambda x: f"${x:,.0f}"))

print("\n=== Fills per wallet (top 15) ===")
print(df["address"].value_counts().head(15))

In [ ]:
# --- Per-wallet summary ---
wallet_summary = df.groupby("address").agg(
    fills        = ("tid",       "count"),
    total_volume = ("notional",  "sum"),
    total_pnl    = ("closedPnl", "sum"),
    total_fees   = ("fee",       "sum"),
    assets_traded= ("asset",     "nunique"),
    first_fill   = ("time",      "min"),
    last_fill    = ("time",      "max"),
    open_longs   = ("direction", lambda x: (x == "Open Long").sum()),
    open_shorts  = ("direction", lambda x: (x == "Open Short").sum()),
).sort_values("total_volume", ascending=False).reset_index()

wallet_summary["long_pct"] = (wallet_summary["open_longs"] / 
    (wallet_summary["open_longs"] + wallet_summary["open_shorts"]).replace(0, pd.NA) * 100).round(1)

wallet_summary["total_volume"] = wallet_summary["total_volume"].apply(lambda x: f"${x:,.0f}")
wallet_summary["total_pnl"]    = wallet_summary["total_pnl"].apply(lambda x: f"${x:,.0f}")
wallet_summary["total_fees"]   = wallet_summary["total_fees"].apply(lambda x: f"${x:,.0f}")

print("Per-wallet summary (top 20 by volume):")
wallet_summary.head(20)